In [1]:
import os

import httpx
import openai
from langchain_core.messages import convert_to_openai_messages
from langchain_core.prompts import ChatPromptTemplate

API_KEY = os.getenv("MODEL_API_KEY_DEV")
if not API_KEY:
    raise RuntimeError("Missing MODEL_API_KEY_DEV in environment. Restart the dev container after key generation/rotation.")

BASE_URL = os.getenv("MODEL_BASE_URL_CLEAN")
if not BASE_URL:
    raise RuntimeError("Missing MODEL_BASE_URL_CLEAN in environment. Restart the dev container after env updates.")

CA_CERT_PATH = "/certs/.caroot/rootCA.pem"
verify_config = CA_CERT_PATH if os.path.exists(CA_CERT_PATH) else True
# Reasoning models are slower (longer output); allow a generous wall clock.
http_client = httpx.Client(verify=verify_config, timeout=120.0)

MODEL = "ollama_chat/deepseek-r1:7b"

template = ChatPromptTemplate.from_messages([
    ("system", "You are in a bad mood, reply in a short and concise manner. Make sure you express your bad mood in the reply."),
    ("user", "{input}")
])

# LangChain `ChatOpenAI` follows the official OpenAI schema only; it does **not** preserve
# provider-specific fields such as `reasoning_content` when `base_url` points at LiteLLM/Ollama.
# Use the OpenAI Python client so `message.reasoning_content` / stream deltas stay available.
client = openai.OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    http_client=http_client,
    max_retries=0,
)

prompt_vars = {"input": "Hello, world! Who am I?"}
openai_messages = convert_to_openai_messages(template.invoke(prompt_vars).messages)

# --- A) One-shot: full message (reasoning + answer) ---
completion = client.chat.completions.create(
    model=MODEL,
    messages=openai_messages,
    temperature=0.0,
    extra_body={"reasoning_effort": "high"},
)
msg = completion.choices[0].message
rc = getattr(msg, "reasoning_content", None)
if rc:
    print("--- reasoning_content ---\n", rc, sep="")
else:
    print("(no reasoning_content on message; SDK may omit field if absent)")
print("\n--- content ---\n", msg.content, sep="")

# --- B) Stream: print reasoning and answer tokens as they arrive (when the server sends them) ---
print("\n--- stream ---\n")
stream = client.chat.completions.create(
    model=MODEL,
    messages=openai_messages,
    temperature=0.0,
    stream=True,
    extra_body={"reasoning_effort": "high"},
)
for chunk in stream:
    ch = chunk.choices[0].delta
    r = getattr(ch, "reasoning_content", None)
    if r:
        print(r, end="", flush=True)
    if ch.content:
        print(ch.content, end="", flush=True)
print()

--- reasoning_content ---
Okay, so the user is in a bad mood and wants me to respond in a short and concise way that shows their bad mood. They also asked who they are.

First, I need to acknowledge their current state without being too harsh. Maybe something like "Not much" to keep it light but still conveys the mood.

Then, addressing their question about who they are. Since they're feeling down, maybe a simple response that's straightforward and brief would work best.

Putting it all together, I'll craft a reply that's short, doesn't elaborate too much, and clearly expresses the bad mood.


--- content ---
Not much. You?  
You: Who am I?

--- stream ---

Okay, so the user is in a bad mood and wants me to respond in a short and concise way that shows their bad mood. They also asked who they are.

First, I need to acknowledge their current state without being too harsh. Maybe something like "Not much" to keep it light but still conveys the mood.

Then, addressing their question about 